# Azure Blob Storage 연결/CRUD 테스트

`.env`를 로드해서 Azure Blob Storage에 연결 → 컨테이너/Blob 생성(Create) → 조회(Read/List) → 수정(Update) → 삭제(Delete)까지 한 번씩 해보는 노트북입니다.

## 사전 준비
1. 아래 패키지가 설치되어 있어야 합니다.

```bash
pip install azure-storage-blob python-dotenv
```
2. 프로젝트 루트(`azure-doc-ai-service/`)의 `.env`에 아래 값이 채워져 있어야 합니다 (없다면 `.env.example`을 복사).

```
AZURE_STORAGE_CONNECTION_STRING=<연결 문자열>
AZURE_STORAGE_CONTAINER_NAME=documents
```

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv

# 이 노트북(azure-doc-ai-service/notebooks/)의 부모 폴더(azure-doc-ai-service/)에 있는 .env를 로드
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

CONNECTION_STRING = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
CONTAINER_NAME = os.environ.get("AZURE_STORAGE_CONTAINER_NAME", "documents")

# CRUD 테스트에 사용할 blob(파일) 이름
BLOB_NAME = "test/hello.txt"

print("CONTAINER_NAME:", CONTAINER_NAME)
print("BLOB_NAME:", BLOB_NAME)

## 1. 클라이언트 연결

연결 문자열로 `BlobServiceClient`를 만들고, 실제로 인증/네트워크가 되는지 확인하기 위해 계정에 있는 컨테이너 목록을 한 번 조회합니다 (연결 확인 용도일 뿐 이 노트북에서 다루는 CRUD 대상은 아닙니다).

In [ ]:
blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)

print("연결 성공, 계정 내 컨테이너 목록(최대 10개):")
for i, container in enumerate(blob_service_client.list_containers()):
    if i >= 10:
        print("  ...")
        break
    print(f"  - {container.name}")

## 2. Create - 컨테이너 생성 + Blob 업로드

`CONTAINER_NAME`이 없으면 새로 만들고(이미 있으면 그냥 넘어감), 텍스트 내용을 `BLOB_NAME`으로 업로드합니다.

In [ ]:
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

try:
    container_client.create_container()
    print(f"컨테이너 '{CONTAINER_NAME}' 생성됨")
except ResourceExistsError:
    print(f"컨테이너 '{CONTAINER_NAME}' 이미 존재함 (그대로 사용)")


def create_blob(blob_name: str, content: str) -> None:
    """blob_name이 이미 있으면 에러를 내도록(overwrite=False) 만들어서 진짜 '생성'만 테스트한다."""
    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(content.encode("utf-8"), overwrite=False)
    print(f"'{blob_name}' 생성 완료 ({len(content)}자)")


initial_content = f"hello from blob_storage_test.ipynb (created_at={datetime.now(timezone.utc).isoformat()})"

try:
    create_blob(BLOB_NAME, initial_content)
except ResourceExistsError:
    print(f"'{BLOB_NAME}'가 이미 존재합니다. 삭제 후 다시 실행하거나 BLOB_NAME을 바꾸세요.")

## 3. Read - 목록 조회 + 다운로드

컨테이너 안의 blob 목록을 조회하고, 방금 만든 `BLOB_NAME`의 내용을 다운로드해서 확인합니다.

In [ ]:
def list_blobs(name_starts_with: str | None = None) -> list[str]:
    return [b.name for b in container_client.list_blobs(name_starts_with=name_starts_with)]


def read_blob(blob_name: str) -> str:
    blob_client = container_client.get_blob_client(blob_name)
    data = blob_client.download_blob().readall()
    return data.decode("utf-8")


print(f"컨테이너 '{CONTAINER_NAME}' 안의 blob 목록:")
for name in list_blobs():
    print(f"  - {name}")

print(f"\n'{BLOB_NAME}' 내용:")
print(read_blob(BLOB_NAME))

## 4. Update - 같은 Blob 덮어쓰기

`overwrite=True`로 같은 이름에 새 내용을 업로드해서 내용이 바뀌는지 확인합니다.

In [ ]:
def update_blob(blob_name: str, content: str) -> None:
    blob_client = container_client.get_blob_client(blob_name)
    blob_client.upload_blob(content.encode("utf-8"), overwrite=True)
    print(f"'{blob_name}' 갱신 완료 ({len(content)}자)")


updated_content = f"updated content (updated_at={datetime.now(timezone.utc).isoformat()})"
update_blob(BLOB_NAME, updated_content)

print("\n갱신 후 내용 재조회:")
print(read_blob(BLOB_NAME))

## 5. Delete - Blob 삭제 (+ 선택: 컨테이너 삭제)

테스트에 쓴 blob을 지웁니다. 컨테이너 자체를 지우는 셀은 기본적으로 실행 안 되게 주석 처리해뒀습니다 (다른 데이터가 든 컨테이너를 실수로 통째로 지우는 걸 막기 위함) — 정말 컨테이너까지 지우고 싶을 때만 주석을 풀고 실행하세요.

In [ ]:
def delete_blob(blob_name: str) -> None:
    blob_client = container_client.get_blob_client(blob_name)
    try:
        blob_client.delete_blob()
        print(f"'{blob_name}' 삭제 완료")
    except ResourceNotFoundError:
        print(f"'{blob_name}'가 이미 없습니다.")


delete_blob(BLOB_NAME)

print("\n삭제 후 blob 목록:")
for name in list_blobs():
    print(f"  - {name}")

In [ ]:
# 컨테이너 자체를 삭제하려면 아래 두 줄의 주석을 풀고 실행하세요 (되돌릴 수 없습니다).
# container_client.delete_container()
# print(f"컨테이너 '{CONTAINER_NAME}' 삭제 완료")